# Week 3: Named Entity Extraction

This notebook shows my Week 3 entity extraction work.

## Deliverables

Regex extraction, taxonomy matching, 200 labeled remarks, evaluation, an 85% F1 target, and error analysis.

In [6]:
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

project_root = Path("..").resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from scripts.entity_extractor import EntityExtractor

print("Project root:", project_root)

Project root: /home/cestame/nlp-internship


## Load the data

In [7]:
cleaned_df = pd.read_csv(
    project_root / "data" / "processed" / "listing_sample_cleaned.csv"
)

labels_df = pd.read_csv(
    project_root / "data" / "processed" / "entity_labeling_template.csv"
)

print("Cleaned listings:", len(cleaned_df))
print("Reviewed labels:", len(labels_df))
print("Columns:", labels_df.columns.tolist())

Cleaned listings: 1000
Reviewed labels: 200
Columns: ['L_ListingID', 'cleaned_remarks', 'suggested_bedrooms', 'gold_bedrooms', 'suggested_bathrooms', 'gold_bathrooms', 'suggested_price', 'gold_price', 'suggested_sqft', 'gold_sqft', 'suggested_amenities', 'gold_amenities', 'reviewed']


## Deliverable 1: EntityExtractor with regex patterns

Regex finds bedrooms, bathrooms, price, and square footage. I included variations such as `bed`, `br`, `bd`, `bath`, and `ba`. I thought price would need phrases such as `priced at`, `cost`, and `listed at`, but the provided rule already searched for large numbers.

In [8]:
extractor = EntityExtractor()

example = (
    "This 3 bedroom home has 2.5 bathrooms, 2000 square feet, "
    "and a community pool. The price is 850000."
)

print(extractor.extract_all(example))

{'bedrooms': 3, 'bathrooms': 2.5, 'price': 850000, 'sqft': 2000, 'amenities': [{'term': 'community pool', 'category': 'community_amenities'}]}


## Try it on real listings

In [9]:
for number, text in enumerate(
    cleaned_df["cleaned_remarks"].head(5),
    start=1,
):
    result = extractor.extract_all(text)
    print(f"\n--- LISTING {number} ---")
    print("Bedrooms:", result["bedrooms"])
    print("Bathrooms:", result["bathrooms"])
    print("Price:", result["price"])
    print("Square feet:", result["sqft"])
    print("Amenities found:", len(result["amenities"]))


--- LISTING 1 ---
Bedrooms: None
Bathrooms: None
Price: None
Square feet: None
Amenities found: 13

--- LISTING 2 ---
Bedrooms: 4
Bathrooms: 2.75
Price: 23971
Square feet: 2223
Amenities found: 10

--- LISTING 3 ---
Bedrooms: None
Bathrooms: None
Price: None
Square feet: None
Amenities found: 11

--- LISTING 4 ---
Bedrooms: 3
Bathrooms: 2.0
Price: None
Square feet: None
Amenities found: 6

--- LISTING 5 ---
Bedrooms: None
Bathrooms: None
Price: None
Square feet: None
Amenities found: 0


## Deliverable 2: Amenity detection using the Week 1 taxonomy

Amenities are matched against the phrases in `taxonomy.json`.

## Deliverable 3: Labeled dataset with entity spans

This checks the 200 labeled remarks and their entity positions.

In [10]:
span_file = (
    project_root / "data" / "processed" / "labeled_entities.jsonl"
)

records = []
label_counts = Counter()

with open(span_file, encoding="utf-8") as file:
    for line in file:
        record = json.loads(line)
        records.append(record)

        for entity in record["entities"]:
            start = entity["start"]
            end = entity["end"]
            assert record["text"][start:end] == entity["text"]
            label_counts[entity["label"]] += 1

print("Records:", len(records))
print("Entity spans:", sum(label_counts.values()))
print("Labels:", dict(label_counts))

Records: 200
Entity spans: 2215
Labels: {'SQFT': 67, 'BEDROOM': 145, 'AMENITY': 1887, 'BATHROOM': 116}


In [11]:
print("Remark preview:")
print(records[0]["text"][:500])

print("\nEntity spans:")
for entity in records[0]["entities"][:10]:
    print(entity)

Remark preview:
Modern Townhome Near Diridon Station, Spacious and well-appointed townhome located in the desirable Cahill Park neighborhood of downtown San Jose. This approximately 1581 square feet residence features 3 bedrooms, 2.5 full bathrooms, plus a flexible office/bonus room or as a 4th bedroom, offering a comfortable layout for modern living. The home includes hardwood flooring, a contemporary kitchen with granite countertops and stainless-steel appliances, and an open living and dining area ideal for 

Entity spans:
{'start': 166, 'end': 182, 'label': 'SQFT', 'text': '1581 square feet'}
{'start': 202, 'end': 212, 'label': 'BEDROOM', 'text': '3 bedrooms'}
{'start': 257, 'end': 267, 'label': 'AMENITY', 'text': 'bonus room'}
{'start': 403, 'end': 422, 'label': 'AMENITY', 'text': 'granite countertops'}
{'start': 478, 'end': 489, 'label': 'AMENITY', 'text': 'dining area'}
{'start': 500, 'end': 515, 'label': 'AMENITY', 'text': 'everyday living'}
{'start': 561, 'end': 585, 'label': 

## Deliverable 4: Precision, recall, and F1 evaluation

Precision checks correctness, recall checks how much was found, and F1 balances both.

In [12]:
metrics_file = project_root / "reports" / "week3_metrics.json"

with open(metrics_file, encoding="utf-8") as file:
    metrics = json.load(file)

metrics_table = pd.DataFrame(metrics).T
metrics_table[["precision", "recall", "f1"]] = (
    metrics_table[["precision", "recall", "f1"]] * 100
)

metrics_table

,tp,fp,fn,precision,recall,f1
bedrooms,100.0,1.0,21.0,99.01,82.64,90.09
bathrooms,47.0,8.0,52.0,85.45,47.47,61.04
price,0.0,15.0,0.0,0.00,0.00,0.00
sqft,49.0,10.0,1.0,83.05,98.00,89.91
amenities,1862.0,0.0,0.0,100.00,100.00,100.00
overall,2058.0,34.0,74.0,98.37,96.53,97.44


## Deliverable 5: Meet the 85% F1 target

Check whether the overall score is at least 85%.

In [13]:
overall_f1 = metrics["overall"]["f1"]

print(f"Overall F1: {overall_f1:.2%}")
print("Meets 85% target:", overall_f1 >= 0.85)

Overall F1: 97.44%
Meets 85% target: True


## Deliverable 6: Error analysis

The main errors came from number words, unusual bathroom formats, and large numbers mistaken for prices. Square footage could also refer to a lot, deck, or garage instead of living space. Bathroom and price extraction need the most improvement.

## Conclusion

The extractor completed all Week 3 deliverables and passed the 85% overall F1 target.